# Module 1 Practice Quiz, Quantum Fundamentals & Environment Setup

This practice quiz draws a **random set of 10 questions** from a bank of **25** each time you start it.

**Course requirement:** you must **complete at least 6 of the 8 module quizzes**. A quiz counts as complete when you score **9/10 or higher (90%)** on it. A colored progress bar shows your score after each submit, and turns green when you complete the module.

You get **2 attempts on each set**. After you submit, only your **score** is shown, not which questions you missed, so you can reconsider and submit once more. On your final attempt the missed question numbers are revealed and the set locks. Click **New questions** to keep half of the current set and swap the other half for new questions.

Your best score for this module is saved automatically. Open **Start_Here** and run the progress cell to see your overall progress across all modules.

This is practice **for your own learning**, the question text is intentionally not selectable/copyable. Please work it out yourself rather than pasting it into an AI tool; you're only quizzing yourself.

> **Requires `ipywidgets`** (standard in JupyterHub). The controls appear when the notebook is *run* in a live kernel; a static preview will look empty.


---
### Setup, run this first


In [ ]:
import json, os, re, random, base64, hashlib, time
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

BANK_PATH = "M1_quiz_bank.json"

with open(BANK_PATH, encoding="utf-8") as _f:
    _BANK = json.load(_f)
SALT = _BANK["salt"]
_QUESTIONS = _BANK["questions"]
_MODULE = _BANK.get("module")
_TITLE = _BANK.get("title", "Module " + str(_MODULE))
# shared progress file at the course root (two levels up from this practices/ folder)
_PROGRESS_PATH = os.path.abspath(os.path.join(os.path.dirname(BANK_PATH), "..", "..", ".quiz_progress.json"))

def _dec(s):
    return base64.b64decode(s.encode("ascii")).decode("utf-8")

def _normalize(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def _is_correct(q, value):
    got = hashlib.sha256((SALT + _normalize(value)).encode()).hexdigest()
    return got == q["answer_hash"]

_GUARD = """
<style>
.qc-noselect, .qc-noselect * {
    -webkit-user-select: none !important;
    -moz-user-select: none !important;
    -ms-user-select: none !important;
    user-select: none !important;
}
.qc-noselect input, .qc-noselect textarea {
    -webkit-user-select: text !important;
    -moz-user-select: text !important;
    -ms-user-select: text !important;
    user-select: text !important;
}
</style>
<script>
(function(){
  function inGuard(n){
    while(n){ if(n.classList && n.classList.contains('qc-noselect')) return true; n=n.parentNode; }
    return false;
  }
  if(!window.__qcGuard){
    window.__qcGuard = true;
    ['copy','cut'].forEach(function(ev){
      document.addEventListener(ev, function(e){
        var s = document.getSelection();
        if(s && s.anchorNode && inGuard(s.anchorNode)) e.preventDefault();
      }, true);
    });
    document.addEventListener('contextmenu', function(e){ if(inGuard(e.target)) e.preventDefault(); }, true);
  }
})();
</script>
"""

_GUARD_DONE = [False]
def _inject_guard():
    if not _GUARD_DONE[0]:
        _GUARD_DONE[0] = True
        display(HTML(_GUARD))

_NOSEL = ("user-select:none;-webkit-user-select:none;"
          "-moz-user-select:none;-ms-user-select:none")

def _lettered(q):
    return [(letter, _dec(label_b64)) for label_b64, letter in q["options"]]

def _prompt_html(i, q):
    parts = [f'<div style="{_NOSEL}"><b>Q{i}.</b> {_dec(q["prompt"])}']
    if q["kind"] == "choice":
        for letter, label in _lettered(q):
            parts.append(f'<br><b>{letter.upper()}.</b> {label}')
    parts.append("</div>")
    return "".join(parts)

def _save_progress(score, total):
    """Record the best score for this module in the shared progress file. Never raises."""
    try:
        data = {}
        if os.path.exists(_PROGRESS_PATH):
            with open(_PROGRESS_PATH, encoding="utf-8") as f:
                data = json.load(f)
        key = str(_MODULE)
        ratio = (score / total) if total else 0.0
        prev = data.get(key)
        prev_ratio = (prev["best_score"] / prev["best_total"]) if (prev and prev.get("best_total")) else -1.0
        if ratio > prev_ratio:
            data[key] = {"module": _MODULE, "title": _TITLE,
                         "best_score": score, "best_total": total,
                         "met": ratio >= 0.9, "updated": time.strftime("%Y-%m-%d %H:%M")}
            with open(_PROGRESS_PATH, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)
    except Exception:
        pass

def _status_html(score, total, used, attempts, final, missed):
    pct = int(round(100 * score / total)) if total else 0
    met = score * 10 >= 9 * total          # 90% or better
    if met:
        bar, bg, bd, fg = "#2e7d32", "#e6f4ea", "#2e7d32", "#1b5e20"
    else:
        bar, bg, bd, fg = "#ef6c00", "#fff3e0", "#ef6c00", "#8a4b00"
    out = [f'<div style="color:#888;margin:4px 0">Attempt {used} of {attempts}</div>']
    out.append(
        '<div style="background:#d0d0d0;border-radius:6px;height:26px;width:100%;max-width:420px;overflow:hidden">'
        f'<div style="width:{max(pct,8)}%;height:100%;background:{bar};color:#fff;font-weight:700;'
        'display:flex;align-items:center;justify-content:center">'
        f'{score} / {total}</div></div>')
    if met:
        msg = "Module complete: you scored 90% or higher on this quiz."
    else:
        msg = "Below 90%. Score 9 out of 10 to complete this module (the course needs 6 of 8 complete)."
    if not final:
        msg += f" You have {attempts - used} more attempt(s) on this set."
    elif missed:
        msg += (" Missed: " + ", ".join("Q" + str(i) for i in missed)
                + ". Answers are not shown; use New questions for a fresh set.")
    else:
        msg += " Perfect score."
    out.append(f'<div style="margin-top:8px;padding:10px 12px;border-radius:6px;background:{bg};'
               f'border:1px solid {bd};color:{fg};font-weight:600;max-width:560px">{msg}</div>')
    return "".join(out)

def start_quiz(n=10, attempts=2):
    """Draw a random set of n questions. Two attempts per set; a colored bar shows your score.

    'New questions' keeps a random HALF of the current set and swaps the other half for new
    questions, so a re-roll is not a completely fresh set. Your best score is saved to
    ../../.quiz_progress.json (Start_Here can show your overall progress).
    """
    n = min(n, len(_QUESTIONS))
    shell = widgets.Output()
    current = list(random.sample(_QUESTIONS, n))   # persists across re-rolls

    def swap_half():
        keep_k = max(1, n // 2)
        keep = random.sample(current, min(keep_k, len(current)))
        keep_ids = {id(q) for q in keep}
        pool = [q for q in _QUESTIONS if id(q) not in keep_ids]
        newq = random.sample(pool, min(n - len(keep), len(pool)))
        combined = keep + newq
        if len(combined) < n:  # tiny bank: top up with anything left
            have = {id(q) for q in combined}
            extra = [q for q in _QUESTIONS if id(q) not in have]
            random.shuffle(extra)
            combined += extra[:n - len(combined)]
        random.shuffle(combined)
        current[:] = combined

    def render():
        with shell:
            clear_output(wait=True)
            _inject_guard()
            fields, rows = [], []
            for i, q in enumerate(current, 1):
                rows.append(widgets.HTML(_prompt_html(i, q)))
                if q["kind"] == "choice":
                    opts = [(letter.upper(), letter) for letter, _label in _lettered(q)]
                    field = widgets.RadioButtons(options=opts, value=None,
                                                 layout=widgets.Layout(width="auto"))
                else:
                    field = widgets.Text(placeholder="type your answer")
                fields.append((q, field))
                rows.append(field)
                rows.append(widgets.HTML("<hr style='opacity:0.3'>"))

            submit = widgets.Button(description="Submit answers", button_style="primary",
                                    layout=widgets.Layout(width="200px"))
            reroll = widgets.Button(description="New questions (swap half)",
                                    layout=widgets.Layout(width="230px"))
            result = widgets.Output()
            used = [0]

            def on_submit(_):
                with result:
                    clear_output(wait=True)
                    missing = [i for i, (q, f) in enumerate(fields, 1)
                               if f.value is None or str(f.value).strip() == ""]
                    if missing:
                        display(HTML('<div style="color:#c62828;font-weight:600">Answer every '
                                     'question before submitting. Missing: '
                                     + ", ".join("Q" + str(i) for i in missing) + "</div>"))
                        return
                    used[0] += 1
                    score = 0
                    missed = []
                    for i, (q, f) in enumerate(fields, 1):
                        if _is_correct(q, f.value):
                            score += 1
                        else:
                            missed.append(i)
                    final = used[0] >= attempts
                    _save_progress(score, len(fields))
                    display(HTML(_status_html(score, len(fields), used[0], attempts, final, missed)))
                    if final:
                        for q, f in fields:
                            f.disabled = True
                        submit.disabled = True

            def on_reroll(_):
                swap_half()
                render()

            submit.on_click(on_submit)
            reroll.on_click(on_reroll)

            box = widgets.VBox(rows + [widgets.HBox([submit, reroll]), result])
            box.add_class("qc-noselect")
            display(box)

    render()
    display(shell)


---
## Take the quiz

Run the cell below. You can change `n` (questions per set) or `attempts` if you like.


In [ ]:
start_quiz(n=10, attempts=2)
